---
# Python API
One `RunModel` instance; call stages in order. You can stop, inspect files, and continue — but note stages 3→5 share paths in memory, so within a *fresh* session start from the stage whose inputs already exist on disk (each stage re-derives what it needs).

## Scenario Configuration [Interactive]

In [ ]:
import bcnexus.utils as utils
(scenarios_box, storage_tg, hg, nc, ts_label,
 solver_bg, compiler_bg, threads_bg) = utils.build_scenario_ui() # note the int() cast

- Ingest values from the UI

In [ ]:
scenarios = utils.get_selected_scenarios(scenarios_box)   # or scenarios_box.value
STORAGE_ALGO  = storage_tg.value # "Kotzur"           # Kotzur | Niet
HOUR_GROUPING = hg.value # 3
N_CLUSTERS    = nc.value # 4
SOLVER        = solver_bg.value.lower()          # gurobi | cbc
COMPILER      = compiler_bg.value.lower()        # cplex | gurobi | xpress | cbc
THREADS       = threads_bg.value # 16
INCLUDE_LIVESTOCK = False  

- Initiate the `RunModel` object

In [ ]:
from bcnexus.clews.runner import RunModel

- Trimmed snapshot for diagnostics purposes
> The ordering matters: RunModel.__init__ → get_all_attributes() → aparser.get_clews_snapshot() captures the value once, so mutating afterwards changes nothing.

In [ ]:
from bcnexus.clews import model_structure as ms
ms.snapshot.update(start=2021, end=2050)# for calibration purposes, we need to run the model until 2035, even if the scenario ends earlier.

# X Full Workflow with Scenario pipeline

In [ ]:
import yaml
cfg = yaml.safe_load(open('config/scenarios_bcnexus.yaml'))['SCENARIOS']
missing = [s for s in scenarios if s not in cfg]
assert not missing, f"Not in scenarios_bcnexus.yaml: {missing}"
print(f"Planned: {len(scenarios)} scenarios x {storage_tg.value} x "
      f"{(24//hg.value)*nc.value} timeslices, solver={solver_bg.value}")

In [ ]:
import time
import traceback
import pandas as pd

results_log = []

  # verify it took effect
for scene in scenarios:
    utils.print_banner(f"Running scenario: {scene}")
    t0 = time.time()
    try:
        m = RunModel(run_scenario=scene,
                     storage_algorithm=storage_tg.value,
                     clustering_attributes={'hour_grouping': hg.value,
                                            'n_clusters': nc.value})
        utils.print_info(f" Snapshot {m.start_year}-{m.last_year}")
        m.run(build=True, include_livestock=False,
              solver_name=solver_bg.value.lower(),
              threads=int(threads_bg.value))
        status, note = "OK", ""
        utils.print_banner(f"Finished scenario: {scene}")

    except KeyboardInterrupt:                 # let Ctrl-C stop the whole batch
        print("Batch interrupted by user.")
        raise

    except Exception as e:
        status, note = "FAILED", f"{type(e).__name__}: {e}"
        utils.print_error(f"X {scene} failed — continuing with next scenario")
        traceback.print_exc()                 # full trace stays in the output

    results_log.append({"scenario": scene,
                        "status": status,
                        "minutes": round((time.time() - t0) / 60, 1),
                        "note": note})

summary = pd.DataFrame(results_log)
display(summary)
print(f"{(summary.status=='OK').sum()} succeeded, "
      f"{(summary.status=='FAILED').sum()} failed, of {len(summary)}")

2026-07-19 15:03:29,628 - INFO -   391146   -2.6241207e+33   6.571888e+37   2.624143e+03   1170s
2026-07-19 15:03:34,707 - INFO -   393267   -2.5494605e+33   6.754141e+37   2.549483e+03   1175s
2026-07-19 15:03:39,506 - INFO -   395691   -2.4898566e+33   5.041010e+37   2.489881e+03   1180s
2026-07-19 15:03:44,481 - INFO -   398418   -2.3811053e+33   5.966469e+37   2.381133e+03   1185s
2026-07-19 15:03:49,498 - INFO -   401145   -2.2428667e+33   1.881162e+38   2.242897e+03   1190s
2026-07-19 15:03:54,457 - INFO -   403771   -2.0993482e+33   5.894902e+37   2.099379e+03   1195s
2026-07-19 15:03:59,589 - INFO -   406498   -1.9464334e+33   2.774687e+37   1.946467e+03   1200s
2026-07-19 15:04:04,562 - INFO -   409528   -1.8407666e+33   2.960589e+37   1.840803e+03   1205s
2026-07-19 15:04:09,445 - INFO -   412558   -1.7514378e+33   7.580880e+37   1.751475e+03   1210s
2026-07-19 15:04:14,440 - INFO -   415790   -1.4236816e+33   1.894480e+37   1.423719e+03   1215s
2026-07-19 15:04:19,520 - INFO

# X  Inspection - individual stage, per scenario

 BCNexus — Stage-wise Workflow explained
----
**Author:** Md Eliasinul Islam

Run the CLEWs pipeline **one stage at a time**, via the Python API or the CLI — both drive the same `RunModel.stage_*` methods, so results are identical.

| Stage | Python | CLI |
|---|---|---|
| 1. Build SETs/params | `m.stage_build(include_livestock=INCLUDE_LIVESTOCK)` | `python -m bcnexus.stages build ...` |
| 2. Scenario overrides | `m.stage_scenario()` | `... scenario-overlay ...` |
| 3. otoole datafile | `m.stage_datafile()` | `... datafile ...` |
| 4. LP file (glpsol) | `m.stage_lp()` | `... lp ...` |
| 5. Solve (+duals/reports) | `m.stage_solve()` | `... solve ...` |
| 6. Result CSVs (otoole) | `m.stage_results()` | `... results ...` |

Why stages: after a crash (or a config tweak) you re-run only the affected step — e.g. edit `scenarios_bcnexus.yaml`, then re-run stages 2→6 without rebuilding.

> Run this notebook from the **repo root** (where `config/` and `data/` live).

## Set a specific scenario

- See all scenarios configured

In [ ]:
# scenarios

- Select a specific scenario

In [ ]:
# scene=scenarios[0]

- Initiate the `RunModel` object with this scenario setup

In [ ]:
# m = RunModel(run_scenario=scene, storage_algorithm=storage_tg.value,
#                  clustering_attributes={'hour_grouping': hg.value, 'n_clusters': nc.value})

## Stage 1 — CLEWs builder: SETs, params, temporal profiles, storage schema

In [ ]:
# built_csvs = m.stage_build(include_livestock=INCLUDE_LIVESTOCK)

## Stage 2 — scenario overrides from config/scenarios_bcnexus.yaml

In [ ]:
# m.stage_scenario()

## Stage 3 — otoole datafile + model file

In [ ]:
# data_file, model_file = m.stage_datafile()
# print(data_file, model_file, sep="\n")

## Stage 4 — LP file via glpsol

- Planned to try MOSOX here

In [ ]:
# lp_file = m.stage_lp()
# lp_file

## Stage 5 — solve (writes .sol + duals, constraints summary, ELCB02 shadow price)

In [ ]:
# sol = m.stage_solve(solver_name=SOLVER, threads=THREADS)
# sol   # None => not optimal; check the solver log in the run dir

## Stage 6 — extract result CSVs via otoole

- <span style="color: grey;"> Check Results @ `results\clews\<Model_<storage_algorithm>_<scenario>`</span>

In [ ]:
results_dir = m.stage_results(solver_name=SOLVER)
results_dir  # => .../{N}ts/result_csvs_gurobi/

- <span style="color: grey;"> We can also use the `datapackage` module's `GetDataPackage` object to load load the results as an object.
- <span style="color: grey;"> This results pack could be used for __post-analysis, result exchange to other models, visualizations, scenario dashes__ etc.

- <span style="color: grey;">  Get `result_pack` as an instance of `GetDataPackage` object

In [ ]:
from bcnexus.clews.datapackage import GetDataPackage
result_pack=GetDataPackage(results_dir)

- <span style="color: grey;">  Apply `get_dataframe(<Result param name>)` method of the `result_pack` instance

## <span style="color: yellow;"> Plots </span>

- <span style="color: biege;">  Load the `plotter` module. 
  - <span style="color: grey;"> plotter.`get_plots()` function returns a dictionary of plots. You can review the plots in this notebook from that dictionary. The function also save the plots as html to 'vis/bccm/< scenario > '

In [ ]:
results_dir.parent

In [ ]:
from pathlib import Path
vis_dir = Path(str(results_dir).replace('results', 'vis', 1))

In [ ]:
results_dir.parent

In [ ]:
run_log=results_dir.parent/'runtime_memory_log.txt'
gurobi_log=results_dir.parent/'gurobi.log'
constraints_log=results_dir.parent/'constraints_summary.txt'

In [ ]:
from pathlib import Path
import bcnexus.plots as plotter

plotter_args=dict(
    nexus_scenario=scene,
    storage_algorithm=STORAGE_ALGO,
    timeslices=m.timeslices,
    results_csvs=results_dir,
    plots_save_to=Path('vis')
)
# plotter.main(**plotter_args)
nexus_plots_dict=plotter.get_plots(**plotter_args,
                              extra_info="Plots are under validation. Please consult to developer before citation") # enable `save_individual`=True for individual plots

- check what's inside the plot dict

In [ ]:
utils.print_banner("Plot categories")
for k1,v1 in nexus_plots_dict.items():
    if k1 not in ["Climate","Land","Energy","Water"]:
            utils.print_info(f"timeslices: {k1}")
    else:
        utils.print_update(level=1,message=k1)
        for k2,v2 in v1.items():
            utils.print_update(level=2,message=k2)
        

### Stable Plots

In [ ]:
nexus_plots_dict['Water']['hydro_water_energy']

In [ ]:
nexus_plots_dict['Energy']['power_generation_annual']

In [ ]:
nexus_plots_dict['Energy']['power_generation_timeslices'] 

### Dev. Plots
!! <span style="color: biege;"> Still under active development to include more comprehensive plots <span style="color: yellow;"> EL_20260719

#### Lands

In [ ]:
import bcnexus.vis.plot_Land as Lvis

In [ ]:
# Lvis.plot_land_area_by_crop (result_pack.get_df('ProductionByTechnologyAnnual'),scene)

-  The intensity line is diagnostic, and here it's telling you something is off. 0.08–0.10 BCM per 1000 km² converts to 80–100 mm of irrigation water per year — real irrigation applications run several hundred mm/yr. Combined with observation 1, the likely story: irrigated land techs in the model carry a low water input requirement (IAR), making irrigation nearly free, so the optimizer picks HI/II everywhere for the yield advantage. Cheap water in → total irrigation dominance out.
-  The plausibility check this demands. Statistics Canada puts BC's actually irrigated farmland at roughly 1–2 thousand km² — an order of magnitude below the ~27 kkm² the model shows. So this plot, doing exactly its job, is flagging that either the water-input coefficients on irrigated land techs or the yield differentials between regimes need re-examination before land–water results are cited. Check InputActivityRatio for LNDxxx HI/II techs → AGRWATBC1, against a crop-water benchmark (e.g., FAO crop water requirements, ~400–700 mm for most temperate crops).
This matters doubly for the Canada scale-up: the Prairies' irrigation question (Alberta's irrigation districts, Lake Diefenbaker expansion) is a live policy issue, and a model that irrigates everything for free would produce confidently wrong answers there. Worth logging as a calibration issue in the project notes — want me to add it to the plan document's verification items?

In [ ]:
Lvis.plot_irrigated_vs_rainfed (result_pack.get_df('ProductionByTechnologyAnnual'),scene)

In [ ]:
# Lvis.plot_landcover_change (result_pack.get_df('TotalCapacityAnnual'),scene)

In [ ]:
# Lvis.plot_cropland_change   (result_pack.get_df('TotalCapacityAnnual'),scene)

In [ ]:
# Lvis.plot_effective_yield (result_pack.get_df('ProductionByTechnologyAnnual'),scene)

In [ ]:
# Lvis.plot_forest_trajectory (result_pack.get_df('TotalCapacityAnnual'),scene)

In [ ]:
# Lvis.plot_cluster_crop_heatmap (result_pack.get_df('TotalAnnualTechnologyActivityByMode'), year=2040)


# modes = [m.strip().strip("'") for m in open("data/clews_data/SETs/ModeList.txt").read().split(",") if m.strip()]
# mode_names = {i + 1: name for i, name in enumerate(modes)}

# bymode = result_pack.get_df('TotalAnnualTechnologyActivityByMode')
# Lvis.plot_cluster_crop_heatmap(bymode, year=2035, mode_names=mode_names)

#### Climate

In [ ]:
import bcnexus.vis.plot_Climate as Cvis
# Cvis.plot_cumulative_emissions(result_pack.get_df('AnnualEmissions'), scene)

In [ ]:
# Cvis.plot_emissions_penalty_cost(result_pack.get_df('DiscountedTechnologyEmissionsPenalty'), scene)

In [ ]:
# Cvis.plot_sector_emission_intensity (result_pack.get_df('AnnualTechnologyEmission'),result_pack.get_df('ProductionByTechnologyAnnual'), scene)

In [ ]:
Cvis.plot_target_gap (result_pack.get_df('AnnualTechnologyEmission'), 0)

In [ ]:
import pandas as pd
from pathlib import Path


root = Path("results/clews")
runs = {
    "REF": pd.read_csv(root / "Model_Kotzur_Base_CNZ/32ts/result_csvs_gurobi/AnnualTechnologyEmission.csv"),
    "CO2 limit": pd.read_csv(root / "Model_Kotzur_CNZ_LIMITED_CO2/32ts/result_csvs_gurobi/AnnualTechnologyEmission.csv"),
    "CO2 penalty": pd.read_csv(root / "Model_Kotzur_CNZ_LIMITED_CO2_PTY/32ts/result_csvs_gurobi/AnnualTechnologyEmission.csv"),
}
fig = Cvis.plot_scenario_emission_wedges(runs, reference="REF")
fig.show()

#### Water

In [ ]:
import bcnexus.vis.plot_Water as Wvis

In [ ]:
# Wvis.plot_water_balance(result_pack.get_df('ProductionByTechnologyAnnual'), scene)

In [ ]:
# Wvis.plot_water_use_by_purpose(result_pack.get_df('ProductionByTechnologyAnnual'), scene)

In [ ]:
# Wvis.plot_reservoir_levels (result_pack.get_df('StorageLevelYearStart'), scene)

In [ ]:
Wvis.plot_hydro_water_energy  (result_pack.get_df('ProductionByTechnologyAnnual'),result_pack.get_df('StorageLevelYearStart'), scene)

#### Energy

In [ ]:
import bcnexus.vis.plot_Energy as Evis

In [ ]:
Evis.plot_system_cost_breakdown(result_pack.get_df('CapitalInvestment'), 
                                result_pack.get_df('AnnualFixedOperatingCost'), 
                                result_pack.get_df('AnnualVariableOperatingCost'), 
                                result_pack.get_df('DiscountedTechnologyEmissionsPenalty'),  
                                scene)

In [ ]:
# Evis.plot_fossil_imports(result_pack.get_df('ProductionByTechnologyAnnual'), scene)


In [ ]:
prod=result_pack.get_df('ProductionByTechnologyAnnual')
use=result_pack.get_df('UseByTechnology')
# Evis.plot_nexus_sankey(prod, use, 2021, scene)
# fig1 = Evis.plot_nexus_sankey_slider(prod, use, scenario='Base_CNZ_noCCS', step=5)
# fig2 = Evis.plot_nexus_sankey_slider(prod, use, years=[2030, 2040, 2050])
fig3 = Evis.plot_nexus_sankey_slider(prod, use, step=1)              # all 30 years

In [ ]:
fig3

# <span style="color: Coral;"> Playground for Solved Model </span>

* <span style="color: grey;">Load the Gurobi Model object as `m` from `clewsRun`'s attribute `solved_model` </span>  

In [ ]:
# m.solved_model.printStats()          # Print basic stats: number of vars, constraints, etc.